# Applied Validation: Power-Flow Stress and Expansion Costs

The joint SWF + ÜZW run (`joint_setup.RUN_ID`) in the status quo and after electrification without HEMS
(INFLEX) and with HEMS, for the synthetic and the real grids. The pooled figure and tables label the providers as
*Real* and *Synthetic*; the per-provider views are diagnostics only.

**Grid set.**
- Power flow: the grids with non-converged hours in at least 1 % of the hours of any case are left out of every
  case of their group (`joint_setup.MIN_FAILED_SHARE`).
- Expansion costs: in addition, every real grid whose cost is incomplete in one of the cases. The materialization
  gives no cost to a real grid with non-converged hours, because its P100 loading is unknown. All cases then cover
  the same grids. Both lists are shown below.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from gridexpand.analysis.expansion.notebook_workflow import (
    PROVIDER_LABELS,
    expansion_cost_comparison,
    expansion_cost_reduction_summary,
    expansion_grid_costs,
    export_scenario_analysis_manifest,
    group_network,
    load_cable_loading_decomposition,
    load_powerflow_cutoff_comparison,
    load_transformer_import_distributions_for_specs,
    load_voltage_summaries_for_powerflow_comparison,
)
from gridexpand.analysis.plotting.expansion_cost_plots import (
    COST_COMPONENT_COLORS,
    plot_expansion_cost_overview_static,
    plot_expansion_cost_stacked_static,
)
from gridexpand.analysis.plotting.powerflow_asset_plots import (
    plot_cable_capacity_current_loading_comparison,
    plot_powerflow_asset_cutoff_overview_static,
)
from gridexpand.analysis.plotting.powerflow_transformer import plot_transformer_import_distributions_matplotlib
from gridexpand.analysis.plotting.powerflow_voltage import plot_voltage_deviation_histogram_comparison
from joint_setup import CASE_COLORS, CASE_DISPLAY, CURVE_Y_LIMITS, OUTPUT_DIR, STAGE_LABELS, prepare_joint_analysis

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

context = prepare_joint_analysis()
CASES = list(STAGE_LABELS.values())
PRE, INFLEX, HEMS = STAGE_LABELS["pre"], STAGE_LABELS["post_inflex"], STAGE_LABELS["post_flex"]
FIGURE_DIR = OUTPUT_DIR / "expansion"
export_scenario_analysis_manifest(context, output_dir=FIGURE_DIR, excluded_grids=context["excluded_grids"])

## Readiness

The gate checks the raw run: `compact power-flow runs complete` fails as long as any grid has non-converged hours,
also the grids excluded below.

In [ ]:
display(context["powerflow_status"])
display(context["publication_gate"])
display(context["analysis_status"])
if not context["publication_ready"]:
    print("DIAGNOSTIC: at least one readiness check failed (see the gate above).")

## Non-converged grids

In [ ]:
display(context["nonconverged"])
context["excluded_grids"]

## Power-flow stress

Figure *paired-asset-cutoff-overview*: mean stress over the retained grids by retained-grid cutoff (top) and the
distribution of each grid's critical asset at the P99 cutoff (bottom). Synthetic (left) and real grids (right) sit side
by side in each case, real lighter with a dashed outline. The dotted line is the status-quo median.

In [ ]:
comparison = load_powerflow_cutoff_comparison(
    specs_by_source=context["specs_by_source"],
    stage_order=CASES,
    excluded_grids=context["excluded_grids"],
)
profile = comparison["profile"]
display(comparison["coverage_summary"].pivot(index="data_source", columns="comparison_stage", values="grids"))

OVERVIEW = dict(
    group_col="comparison_stage",
    source_col="network",
    color_map=CASE_COLORS,
    group_labels=CASE_DISPLAY,
    reference_group=PRE,
    asset_cutoff_percentile=0.99,
    asset_percentiles=(0.10, 0.25, 0.50, 0.75, 0.90, 1.0),
    filter_scope="grid",
    worst_asset_per_grid=True,
    curve_y_axis_limits=CURVE_Y_LIMITS,
    distribution_y_axis_limits={"Voltage": (0.80, None)},
)
fig = plot_powerflow_asset_cutoff_overview_static(
    profile, save_path=FIGURE_DIR / "pre_post_flex_no_flex_asset_cutoff_overview", **OVERVIEW
)
plt.show()

### Per provider (diagnostic; names the providers)

In [ ]:
for provider in context["providers"]:
    label = PROVIDER_LABELS[provider]
    fig = plot_powerflow_asset_cutoff_overview_static(
        profile[profile["provider"] == label],
        title=label,
        save_path=FIGURE_DIR / f"pre_post_flex_no_flex_asset_cutoff_overview_{provider}",
        **OVERVIEW,
    )
    plt.show()

## Expansion costs

Staged LV rules (`de_lv_staged_2026`) per grid, summed over the common grid set of all cases. Figure
*expansion-cost-overview*: total per case with the HEMS change against INFLEX (left) and the cost per asset
in both electrified cases (right; the status quo costs nothing per asset), synthetic solid, real lighter with a
dashed outline. *Other*: load transfers to a neighbouring station with spare capacity (20 k€ per grid) and
voltage-regulating transformers (rONT) for buses still below 0.90 pu.

In [ ]:
grid_costs = expansion_grid_costs(context["expansion_tables_by_source"], CASES)
if grid_costs.empty:
    print("No expansion analyses of this run yet.")
else:
    costs = expansion_cost_comparison(grid_costs, stage_labels=CASES, excluded_grids=context["excluded_grids"])
    costs_by_provider = expansion_cost_comparison(
        grid_costs, stage_labels=CASES, excluded_grids=context["excluded_grids"], by="data_source"
    )
    display(costs["excluded"])
    display(costs["grids"].pivot(index="data_source", columns="stage", values=["grids", "total_cost_eur"]))
    fig = plot_expansion_cost_overview_static(
        costs["costs"],
        stage_order=CASES,
        asset_stage_order=(INFLEX, HEMS),
        color_map=CASE_COLORS,
        stage_labels=CASE_DISPLAY,
        reduction_stages=(INFLEX, HEMS),
        save_path=FIGURE_DIR / "expansion_cost_overview",
    )
    plt.show()
    display(expansion_cost_reduction_summary(costs["costs"], post_inflex_label=INFLEX, post_flex_label=HEMS))
    display(costs["reinforcements"])

### Variant: stacked totals

The total bars of the figure above without the status quo (almost no cost), each stacked by asset in shades of
the case color (cables at the bottom, *Other* at the top); each legend swatch shows the INFLEX (left) and HEMS
shade (right) of its asset. The dotted boxes are the cost saved through HEMS against INFLEX of the same grids.

In [ ]:
if not grid_costs.empty:
    fig = plot_expansion_cost_stacked_static(
        costs["costs"],
        stage_order=(INFLEX, HEMS),
        color_map=CASE_COLORS,
        stage_labels=CASE_DISPLAY,
        reduction_stages=(INFLEX, HEMS),
        component_colors=COST_COMPONENT_COLORS,
        save_path=FIGURE_DIR / "expansion_cost_stacked",
    )
    plt.show()

### Per provider (diagnostic)

In [ ]:
if not grid_costs.empty:
    display(expansion_cost_reduction_summary(costs_by_provider["costs"], post_inflex_label=INFLEX, post_flex_label=HEMS))
    display(costs_by_provider["grids"].pivot(index="data_source", columns="stage", values=["grids", "total_cost_eur"]))

## Cable capacity and transported current

Whether a loading difference comes from installed ampacity, from the transported peak current, or from both.

In [ ]:
cables = load_cable_loading_decomposition(
    specs_by_source=context["specs_by_source"], excluded_grids=context["excluded_grids"]
)
pooled_cables = cables.assign(data_source=cables["network"])
display(
    pooled_cables.groupby(["data_source", "comparison_stage"], observed=True)
    .agg(
        cables=("asset_id", "count"),
        grids=("grid", "nunique"),
        median_capacity_a=("installed_capacity_a", "median"),
        median_peak_current_a=("max_current_a", "median"),
        median_max_loading_percent=("max_loading_percent", "median"),
    )
    .round(1)
)
plot_cable_capacity_current_loading_comparison(
    pooled_cables,
    stage_order=tuple(CASES),
    color_map=CASE_COLORS,
    save_path=FIGURE_DIR / "cable_capacity_current_loading",
)
plt.show()

## Voltage extremes

In [ ]:
voltage = load_voltage_summaries_for_powerflow_comparison(
    specs_by_source=context["specs_by_source"], excluded_grids=context["excluded_grids"]
)
pooled_voltage = {}
for source, by_case in voltage.items():
    for case, frame in by_case.items():
        pooled_voltage.setdefault(group_network(source), {}).setdefault(case, []).append(frame)
pooled_voltage = {
    network: {case: pd.concat(frames, ignore_index=True) for case, frames in by_case.items()}
    for network, by_case in pooled_voltage.items()
}
plot_voltage_deviation_histogram_comparison(pooled_voltage, lower_limit=0.90, upper_limit=1.10, show=False)

## Transformer import (synthetic grids only)

The real-grid summaries store no signed transformer-import time series.

In [ ]:
frames = []
for source, specs in context["specs_by_source"].items():
    if group_network(source) == "Synthetic":
        distribution = load_transformer_import_distributions_for_specs(specs)
        if not distribution.empty:
            frames.append(distribution.assign(data_source=source))
if not frames:
    print("No transformer-import time series: this run writes compact summaries only.")
else:
    fig = plot_transformer_import_distributions_matplotlib(
        pd.concat(frames, ignore_index=True), metrics=("p", "q", "s"), group_col="comparison_stage", show=False
    )
    fig.savefig(FIGURE_DIR / "transformer_import_distributions.svg", bbox_inches="tight")
    plt.show()